In [1]:
import os
import tempfile
import time
import math
from pathlib import Path

# Keep JAX from preallocating all GPU memory.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Keep Catalyst/JAX temp files on project disk by default.
# You can override root path via CATALYST_TMPDIR.
tmp_root = Path(os.environ.get("CATALYST_TMPDIR", str(Path.cwd())))
tmp_dir = tmp_root / "catalyst_tmp_cache"
tmp_dir.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(tmp_dir)
tempfile.tempdir = str(tmp_dir)

import numpy as np
import scipy.sparse as sp
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import optax
import pennylane as qml
import catalyst


In [2]:
print("JAX version:", jax.__version__)
print("Devices:", jax.devices())
use_gpu = any(d.platform == "gpu" for d in jax.devices())
print("GPU detected:", use_gpu)
if not use_gpu:
    print("Warning: lightning.gpu requires a GPU-enabled runtime.")


JAX version: 0.6.2
Devices: [CudaDevice(id=0)]
GPU detected: True


In [3]:
# 1. Import required libraries (same as before; must use scipy.sparse)
import os
import scipy.sparse as sp
from pathlib import Path
import re

# 2. Automatically locate matrix files (supports N/n case differences and different startup directories)
target_l, target_n = 6, 4
matrix_name_candidates = [
    f"L={target_l} N={target_n}.npz",
    f"L={target_l} n={target_n}.npz",
    f"l={target_l} N={target_n}.npz",
    f"l={target_l} n={target_n}.npz",
]
cwd = Path.cwd()

def _normalize_filename(name: str) -> str:
    return "".join(name.lower().split())

matrix_file = None
matrix_override = os.environ.get("MATRIX_FILE")
if matrix_override:
    override_path = Path(matrix_override).expanduser()
    if override_path.exists():
        matrix_file = override_path
    else:
        print(f"⚠️ MATRIX_FILE does not exist: {override_path}")

candidate_roots = [cwd, cwd / "lunwen", cwd.parent, cwd.parent / "lunwen"]
matrix_candidates = [root / name for root in candidate_roots for name in matrix_name_candidates]
if matrix_file is None:
    matrix_file = next((p for p in matrix_candidates if p.exists()), None)

search_roots = [cwd, cwd.parent, cwd.parent.parent]
if matrix_file is None:
    for root in search_roots:
        if not root.exists():
            continue
        for name in matrix_name_candidates:
            matches = sorted(root.rglob(name))
            if matches:
                matrix_file = matches[0]
                break
        if matrix_file is not None:
            break

if matrix_file is None:
    expected_norms = {_normalize_filename(name) for name in matrix_name_candidates}
    fuzzy_hits = []
    npz_inventory = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*.npz"):
            npz_inventory.append(p)
            name_norm = _normalize_filename(p.name)
            if name_norm in expected_norms:
                fuzzy_hits.append(p)
                continue
            has_l = re.search(rf"l\\s*=\\s*{target_l}\\b", p.name, flags=re.IGNORECASE)
            has_n = re.search(rf"n\\s*=\\s*{target_n}\\b", p.name, flags=re.IGNORECASE)
            if has_l and has_n:
                fuzzy_hits.append(p)

    if fuzzy_hits:
        matrix_file = sorted(set(fuzzy_hits), key=lambda x: (len(str(x)), str(x)))[0]
    else:
        tried = [str(p) for p in matrix_candidates]
        sample = [str(p) for p in sorted(set(npz_inventory), key=lambda x: str(x))[:20]]
        scanned = [str(p) for p in search_roots]
        raise FileNotFoundError("Cannot find matrix file for L=6,N=4. cwd={cwd}, tried={tried}, scanned={scanned}, npz_sample={sample}")

H_sparse = sp.load_npz(str(matrix_file))
print(f"Using matrix file: {matrix_file}")

print("Matrix shape:", H_sparse.shape, "nnz:", H_sparse.nnz)

exact_ground_energy = float(
    eigsh(H_sparse, k=1, which="SA", return_eigenvectors=False)[0]
)
print(f"Exact ground energy: {exact_ground_energy:.10f}")

d = H_sparse.shape[0]
n_qubits = int(np.ceil(np.log2(d)))
dim = 2 ** n_qubits
depth = math.ceil(dim / n_qubits) + n_qubits
print(f"Qubits: {n_qubits}, padded dim: {dim}, depth: {depth}, params: {depth * n_qubits}")

H_coo = H_sparse.tocoo()
rows = H_coo.row.astype(np.int64, copy=False)
cols = H_coo.col.astype(np.int64, copy=False)
data = H_coo.data

if dim > d:
    pad = np.arange(d, dim, dtype=np.int64)
    rows = np.concatenate([rows, pad])
    cols = np.concatenate([cols, pad])
    data = np.concatenate([data, np.full(dim - d, 1000.0, dtype=data.dtype)])

def gray_permutation(n_bits: int) -> np.ndarray:
    idx = np.arange(2 ** n_bits, dtype=np.int64)
    return idx ^ (idx >> 1)

gray2natural = gray_permutation(n_qubits)
new_rows = gray2natural[rows]
new_cols = gray2natural[cols]

H_gray = coo_matrix((data, (new_rows, new_cols)), shape=(dim, dim)).tocsr()
H_dense = H_gray.toarray().astype(np.complex128, copy=False)
H_jax = jnp.asarray(H_dense, dtype=jnp.complex128)
print(f"Dense Hamiltonian memory: {H_dense.nbytes / 1024 / 1024:.2f} MiB")

del H_coo, H_gray, H_dense, rows, cols, data, new_rows, new_cols, gray2natural


Matrix shape: (11424, 11424) nnz: 548306
Exact ground energy: -28.0806195146
Qubits: 14, padded dim: 16384, depth: 1185, params: 16590
Dense Hamiltonian memory: 4096.00 MiB


In [4]:
hf_state = np.zeros(n_qubits, dtype=int)
wires = list(range(n_qubits))
dev = qml.device("lightning.gpu", wires=n_qubits)

@qml.qnode(dev, diff_method="adjoint")
def cost_circuit(params_2d, h_mat):
    qml.BasisState(hf_state, wires=wires)
    for w in wires:
        qml.Hadamard(wires=w)

    for layer in range(depth):
        for w in wires:
            qml.RY(params_2d[layer, w], wires=w)

        for w in range(0, n_qubits - 1, 2):
            qml.CNOT(wires=[w, w + 1])

        qml.CNOT(wires=[n_qubits - 1, 0])

        for w in range(1, n_qubits - 1, 2):
            qml.CNOT(wires=[w, w + 1])

    return qml.expval(qml.Hermitian(h_mat, wires=wires))

init_params = jnp.zeros((depth, n_qubits), dtype=jnp.float64)

lr_schedule = optax.exponential_decay(
    init_value=0.01,
    transition_steps=600,
    decay_rate=0.85,
)
optimizer = optax.adam(learning_rate=lr_schedule)
opt_state = optimizer.init(init_params)

@qml.qjit(autograph=True)
def update_step(params, state, h_mat):
    def loss_fn(p):
        return cost_circuit(p, h_mat)

    energy, grads = catalyst.value_and_grad(loss_fn)(params)
    updates, next_state = optimizer.update(grads, state, params)
    next_params = optax.apply_updates(params, updates)
    return next_params, next_state, energy


In [5]:
import platform
import subprocess
import shutil

print("===== Runtime =====")
print("Python:", platform.python_version())
print("JAX:", jax.__version__)
print("PennyLane:", qml.__version__)
print("Catalyst:", catalyst.__version__)
print("JAX devices:", jax.devices())
print("Selected qml device:", dev.name)
print("QNode diff method:", getattr(cost_circuit, "diff_method", "unknown"))
print("Backend: qjit + dense Hermitian")

print("\n===== Problem size =====")
print("H_sparse shape:", H_sparse.shape, "nnz:", H_sparse.nnz, "dtype:", H_sparse.dtype)
print("H_jax shape:", H_jax.shape, "dtype:", H_jax.dtype)
h_bytes = H_jax.size * H_jax.dtype.itemsize
print(f"H_jax memory: {h_bytes / 1024 / 1024:.2f} MiB")
print("n_qubits:", n_qubits, "depth:", depth, "params:", depth * n_qubits)

ry_per_layer = n_qubits
cnot_per_layer = len(range(0, n_qubits - 1, 2)) + 1 + len(range(1, n_qubits - 1, 2))
total_gates = depth * (ry_per_layer + cnot_per_layer) + n_qubits
print("Approx gates per forward:", total_gates)

print("\n===== TMPDIR =====")
print("TMPDIR env:", os.environ.get("TMPDIR"))
print("tempfile dir:", tempfile.gettempdir())

if shutil.which("nvidia-smi"):
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,utilization.gpu,utilization.memory",
                "--format=csv,noheader",
            ],
            text=True,
            stderr=subprocess.STDOUT,
        )
        print("\n===== nvidia-smi =====")
        print(out)
    except Exception as exc:
        print("nvidia-smi query failed:", exc)
else:
    print("nvidia-smi not found in PATH")


===== Runtime =====
Python: 3.12.12
JAX: 0.6.2
PennyLane: 0.43.1
Catalyst: 0.13.0
JAX devices: [CudaDevice(id=0)]
Selected qml device: lightning.gpu
QNode diff method: adjoint
Backend: qjit + dense Hermitian

===== Problem size =====
H_sparse shape: (11424, 11424) nnz: 548306 dtype: float64
H_jax shape: (16384, 16384) dtype: complex128
H_jax memory: 4096.00 MiB
n_qubits: 14 depth: 1185 params: 16590
Approx gates per forward: 33194

===== TMPDIR =====
TMPDIR env: /mnt/data/lzs/miniproject/lunwen/catalyst_tmp_cache
tempfile dir: /mnt/data/lzs/miniproject/lunwen/catalyst_tmp_cache

===== nvidia-smi =====
NVIDIA A100 80GB PCIe, 81920 MiB, 18146 MiB, 0 %, 0 %



In [6]:
print("===== update_step benchmark =====")
diag_params = init_params
diag_state = opt_state

print("Running step1 (compile+run)...")
t0 = time.time()
diag_params, diag_state, e1 = update_step(diag_params, diag_state, H_jax)
e1 = float(e1)
t1 = time.time()
print(f"step1 compile+run : {t1 - t0:.4f}s, energy={e1:.10f}")

print("Running step2 (run only)...")
diag_params, diag_state, e2 = update_step(diag_params, diag_state, H_jax)
e2 = float(e2)
t2 = time.time()
print(f"step2 run only    : {t2 - t1:.4f}s, energy={e2:.10f}")

print("Running step3 (run only)...")
diag_params, diag_state, e3 = update_step(diag_params, diag_state, H_jax)
e3 = float(e3)
t3 = time.time()
print(f"step3 run only    : {t3 - t2:.4f}s, energy={e3:.10f}")

steady_avg = ((t2 - t1) + (t3 - t2)) / 2
planned_steps = globals().get("steps", 5000)
print(f"steady-step avg   : {steady_avg:.4f}s")
print(f"estimated total for {planned_steps} steps: {steady_avg * planned_steps / 60:.2f} min (excluding first compile)")


===== update_step benchmark =====
Running step1 (compile+run)...
step1 compile+run : 55.3635s, energy=309.6289975487
Running step2 (run only)...
step2 run only    : 50.5380s, energy=748.7675981811
Running step3 (run only)...
step3 run only    : 50.4532s, energy=746.1933756978
steady-step avg   : 50.4956s
estimated total for 5000 steps: 4207.97 min (excluding first compile)


In [ ]:
steps = 5000
tolerance = 1e-6

params = init_params
step_history = []
energy_history = []

start = time.time()
for step in range(steps):
    params, opt_state, energy = update_step(params, opt_state, H_jax)
    current = float(energy)

    step_history.append(step)
    energy_history.append(current)

    if step % 50 == 0:
        delta = current - exact_ground_energy
        print(f"step={step:4d} energy={current:.10f} delta={delta:.3e}")

    if abs(current - exact_ground_energy) < tolerance:
        print(f"Converged at step {step} with energy {current:.10f}")
        break

elapsed = time.time() - start
final_energy = energy_history[-1]
print(f"Elapsed: {elapsed:.2f}s")
print(f"Final energy: {final_energy:.10f}")
print(f"Final absolute error: {abs(final_energy - exact_ground_energy):.3e}")


step=   0 energy=309.6289975487 delta=3.377e+02
step=  50 energy=41.4442661132 delta=6.952e+01
step= 100 energy=-11.7493655002 delta=1.633e+01
step= 150 energy=-19.2062366433 delta=8.874e+00
step= 200 energy=-22.8637711199 delta=5.217e+00
step= 250 energy=-25.0856721126 delta=2.995e+00
step= 300 energy=-26.4114765809 delta=1.669e+00
step= 350 energy=-26.9066971836 delta=1.174e+00
step= 400 energy=-26.2646685608 delta=1.816e+00
step= 450 energy=-27.3855878424 delta=6.950e-01
step= 500 energy=-27.3006461631 delta=7.800e-01
step= 550 energy=-27.5387902982 delta=5.418e-01
step= 600 energy=-27.2661946224 delta=8.144e-01
step= 650 energy=-27.3262515557 delta=7.544e-01
step= 700 energy=-27.0789239567 delta=1.002e+00
step= 750 energy=-27.6961540521 delta=3.845e-01
step= 800 energy=-27.7426649528 delta=3.380e-01
step= 850 energy=-27.4451241170 delta=6.355e-01
step= 900 energy=-27.6490345800 delta=4.316e-01
step= 950 energy=-27.6359315876 delta=4.447e-01
step=1000 energy=-27.7332820387 delta=3.4

In [ ]:
import matplotlib.pyplot as plt

if not energy_history:
    raise RuntimeError("No training history recorded.")

plt.figure(figsize=(8, 4))
plt.plot(step_history, energy_history, label="VQE energy")
plt.axhline(
    exact_ground_energy,
    color="tab:red",
    linestyle="--",
    label="Exact ground energy",
)
plt.xlabel("Step")
plt.ylabel("Energy (Ha)")
plt.title("L=6, N=4 VQE convergence")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
